# Estruturação com LLMs (`labdados.estruturacao`)

<a href="https://colab.research.google.com/github/lab-dados/labdados-sdk/blob/main/examples/notebooks/estruturacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Extrai campos estruturados (JSON) de cada documento. Útil para virar uma planilha de ementas/sentenças em colunas.

Modos cobertos aqui:

1. **Nuvem** — usa `gpt-4.1-mini` no Azure OpenAI do escritório (precisa de API key labdados).
2. **Local com OpenAI** — usa sua própria chave da OpenAI rodando no Colab.

Cada célula de execução é independente: rode só o cenário que te interessa. As células de instalação, planilha de exemplo e schema são pré-requisitos compartilhados.

📘 Doc completa: <https://lab-dados.github.io/labdados-sdk>

## 1. Instalar

In [ ]:
%pip install -q 'labdados[estruturacao]'

## 2. Gerar uma planilha de exemplo

No uso real, você apontaria para um CSV/XLSX seu. Aqui criamos uma amostra com 3 ementas fictícias.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'id': [1, 2, 3],
    'ementa': [
        'APELAÇÃO CÍVEL. Plano de saúde. TJSP, 2022. Negativa de cobertura para tratamento '
        'oncológico. Procedência. Cobertura obrigatória. Recurso desprovido.',
        'AGRAVO DE INSTRUMENTO. Direito do consumidor. TJRJ, 2021. Cobrança indevida em fatura '
        'de telefonia. Improcedência. Ausência de prova. Recurso provido para julgar improcedente.',
        'APELAÇÃO. Família. TJMG, 2023. Pedido de guarda compartilhada. Procedente em parte. '
        'Melhor interesse da criança. Manutenção parcial.',
    ],
})
df.to_csv('ementas.csv', index=False)
df

## 3. Definir o schema da extração

O schema é JSON Schema padrão. **Inclua `description` em cada campo** — o LLM lê e melhora a extração.

In [ ]:
schema = {
    'type': 'object',
    'properties': {
        'tribunal': {
            'type': 'string',
            'description': 'Sigla do tribunal (TJSP, TJRJ, STJ...)',
        },
        'ano': {'type': 'integer'},
        'tema': {
            'type': 'string',
            'description': 'Direito do consumidor, família, saúde...',
        },
        'procedente': {
            'type': 'boolean',
            'description': 'Se o pedido principal foi julgado procedente',
        },
    },
    'required': ['tribunal', 'ano', 'tema'],
}

## 4. Modo nuvem

Cole sua API key do labdados quando solicitado (peça uma em <https://labdados-frontend.livelydesert-3e3e3dd8.brazilsouth.azurecontainerapps.io/consultoria/api-key>).

In [ ]:
import glob
import json
import os
import zipfile
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

labdados.estruturacao(
    arquivos='ementas.csv',
    coluna_texto='ementa',     # qual coluna contém o texto
    schema=schema,
    prompt_sistema=(
        'Você está extraindo metadados de ementas de acórdãos brasileiros. '
        'Se o tema não for claro, use "desconhecido".'
    ),
    api_key=os.environ['LABDADOS_API_KEY'],
    saida='extraido_nuvem/',
    modelo='gpt-4.1-mini',
)

# Inspecionar o resultado
for zpath in glob.glob('extraido_nuvem/*.zip'):
    with zipfile.ZipFile(zpath) as z:
        z.extractall('extraido_nuvem/')

for jpath in glob.glob('extraido_nuvem/*.json'):
    print('---', jpath)
    print(json.dumps(json.load(open(jpath)), indent=2, ensure_ascii=False)[:800])

## 5. Modo local com sua chave da OpenAI

Se preferir rodar 100% local (sem API key labdados), use `local=True` apontando para a OpenAI ou qualquer endpoint OpenAI-compatible (Azure OpenAI, vLLM, LM Studio).

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

labdados.estruturacao(
    arquivos='ementas.csv',
    coluna_texto='ementa',
    schema=schema,
    local=True,
    base_url_local='https://api.openai.com/v1',
    api_key_local=os.environ['OPENAI_API_KEY'],
    modelo_local='gpt-4.1-mini',
    saida='extraido_local/',
)
!ls extraido_local/

## Dicas para o schema

- **`description`** em cada campo melhora muito a extração.
- **`required`** apenas para campos sempre presentes — opcionais podem retornar `null`.
- **Enums** funcionam para classificação:
  ```python
  {'type': 'string', 'enum': ['procedente', 'improcedente', 'parcial']}
  ```
- **Listas tipadas**: `{'type': 'array', 'items': {'type': 'string'}}`.

Doc completa dos parâmetros: <https://lab-dados.github.io/labdados-sdk/api/estruturacao.html>